# FHR-DQN on the Atari-100k protocol

**Protocol** (references + reference scores: `src/analysis/atari100k.py`): 100,000
agent-environment interactions per run (400k frames, frameskip 4), deterministic ALE
(`repeat_action_probability: 0.0` — no sticky actions, minimal action set), sign-clipped
rewards for **training** only, then a separate final-policy evaluation of **32 episodes at
ε = 0.001** reporting **raw game scores**. Human-normalised score (HNS) =
(agent − random) / (human − random). Published numbers are quoted verbatim from
EfficientZero (Ye et al. 2021, [arXiv:2111.00210](https://arxiv.org/abs/2111.00210), Table 1);
random/human references are the standard Wang et al. (2016) 30-no-op table.

**Read the caveats before quoting any figure:**
- Every game of the **official 26-game Atari-100k suite** has its own experiment dir
  (`experiments/atari/dqn_<game>/`) with published per-game baselines; **Enduro is an
  extra game OUTSIDE the suite** — no published 100k baselines exist, it reports against
  random/human only and never enters the aggregate. Which suite games feed the aggregate
  figure is selected per game config (`experiment.include_in_aggregate`); **any aggregate
  over fewer than all 26 games is our subset**, never "the Atari-100k benchmark" — the
  figure titles state the games covered.
- Arms: `baseline` = double DQN (`fhr_weight: 0.0`), `exp<N>` = the same agent + the FHR
  penalty set from `experiment.fhr_experiments[N]` — the config diff is the FHR block alone.
  Both train at replay ratio 1 with DER-style settings; they are *each other's* controls.
  Published methods use their own (heavier) recipes, so they contextualise the axis rather
  than being ablations of ours.
- Seeds per arm come from each game config (`experiment.seeds`, default 3 like
  EfficientZero; 5+ is better if budget allows). Score = mean over seeds of the mean over
  32 evaluation episodes.


## Launch — games × arms × seeds, parallel subprocesses

Edit `LAUNCH_GAMES` in the setup cell to choose which games to train (the whole
suite + Enduro is `sorted(GAME_DIRS)`). One subprocess per (game, arm, seed); completed
runs are recorded in each game dir's `cached/fhrdqn100k_runs_manifest.json` (its own
manifest *family*, so the result viewer never pools these with non-100k runs) and are
skipped on relaunch. Logs land in `<game>/cached/logs/fhrdqn100k_<arm>_seed<N>.log`.
A 100k-step Atari run is hours, not minutes — launch and come back, then re-run the
setup cell so the analysis picks up the new games.

In [1]:
import json, pathlib, sys, warnings
# NaN-padded seed grids legitimately have all-NaN columns at the edges
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
import matplotlib.pyplot as plt
import numpy as np

EXPS = pathlib.Path.cwd().parent / "src"          # experiments/src
SRC = pathlib.Path.cwd().parents[1] / "src"       # repo library code
for p in (str(EXPS), str(SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

from run_fhrdqn_atari100k import (launch_all, load_runs, arms_of, GAME_DIRS,
                                  aggregate_games, config_game_key,
                                  _game_dir, _load_manifest)
from analysis.atari100k import (REFERENCE, BASELINES, BENCHMARK_GAMES,
                                ATARI100K_ENV_STEPS, hns, game_key)

# Games to TRAIN when the launch cell below runs. Every suite game has its own
# dir + config; the full 26-game suite + Enduro is sorted(GAME_DIRS).
LAUNCH_GAMES = ["pacman", "seaquest", "enduro"]

# Games the ANALYSIS below covers: everything with at least one recorded run.
GAMES = [g for g in sorted(GAME_DIRS) if _load_manifest(_game_dir(g))["runs"]]
GAME_KEYS = {g: config_game_key(g) for g in GAMES}

# Games feeding the aggregate-HNS figure: each game config opts in/out via
# experiment.include_in_aggregate (suite games only — Enduro can never enter).
AGG_GAMES = aggregate_games(GAMES)
print(f"analysing {len(GAMES)} game(s) with runs: {GAMES}")
print(f"aggregate covers {len(AGG_GAMES)} of {len(BENCHMARK_GAMES)} suite games: {AGG_GAMES}")

analysing 4 game(s) with runs: ['enduro', 'krull', 'pacman', 'seaquest']
aggregate covers 1 of 26 suite games: ['krull']


In [2]:
manifests = launch_all(games=LAUNCH_GAMES, max_workers=3)
{g: sorted(m["runs"]) for g, m in manifests.items()}

skipping 12 already-completed run(s)


{'pacman': ['baseline', 'exp1'],
 'seaquest': ['baseline', 'exp1'],
 'enduro': ['baseline', 'exp1']}

## Results table — EfficientZero-style

Rows are games; columns are random/human, the published Atari-100k methods, and this
repo's arms (mean over seeds of the 32-episode evaluation mean; ± is the std across
seeds). `—` marks games outside the published tables (Enduro).

In [3]:
RESULTS = {}      # game -> arm -> {"means": per-seed eval means, "mean": .., "std": ..}
for g in GAMES:
    RESULTS[g] = {}
    for arm in arms_of(g):
        runs = load_runs(g, arm)
        means = np.array([r["eval_summary"]["mean"] for r in runs])
        RESULTS[g][arm] = {"runs": runs, "means": means,
                           "mean": float(means.mean()), "std": float(means.std())}

OUR_ARMS = sorted({a for g in GAMES for a in RESULTS[g]},
                  key=lambda a: (a != "baseline", a))
OUR_LABEL = {a: ("DQN (baseline)" if a == "baseline" else f"FHR-DQN {a}") for a in OUR_ARMS}
PUBLISHED = ["SimPLe", "OTRainbow", "CURL", "DrQ", "SPR", "MuZero", "EfficientZero"]

hdr = ["Game", "Random", "Human", *PUBLISHED, *[OUR_LABEL[a] for a in OUR_ARMS]]
rows = []
for g in GAMES:
    k = GAME_KEYS[g]
    row = [k, f"{REFERENCE[k]['random']:.1f}", f"{REFERENCE[k]['human']:.1f}"]
    row += [f"{BASELINES[k][m]:.1f}" if m in BASELINES[k] else "—" for m in PUBLISHED]
    row += [f"{RESULTS[g][a]['mean']:.1f} ±{RESULTS[g][a]['std']:.0f}" if a in RESULTS[g]
            else "—" for a in OUR_ARMS]
    rows.append(row)
w = [max(len(r[i]) for r in [hdr] + rows) for i in range(len(hdr))]
for r in [hdr] + rows:
    print("  ".join(s.rjust(w[i]) for i, s in enumerate(r)))

RuntimeError: no completed krull/exp1 run for seed 1 — launch_all() it first (cached/logs/ has failures)

## Human-normalised score per game

One panel per game: published methods (grey), our arms (colour). Dashed line = human
(HNS 1.0), dotted = random (0.0). Value labels carry the exact HNS — on Seaquest every
100k method sits far below human, so labels matter more than bar heights. Enduro shows
our arms only (no published 100k numbers exist).

In [ ]:
FIGDIR = pathlib.Path.cwd() / "cached" / "atari100k_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

def save_and_show(fig, slug):
    fig.savefig(FIGDIR / f"{slug}.png", dpi=200, bbox_inches="tight")
    # also drop a copy into every involved run dir so the result viewer shows it
    for g in GAMES:
        for arm in RESULTS[g]:
            for r in RESULTS[g][arm]["runs"]:
                d = pathlib.Path(r["run_dir"]) / "figures"
                d.mkdir(exist_ok=True)
                fig.savefig(d / f"comparison_{slug}.png", dpi=150, bbox_inches="tight")
    plt.show()

OUR_COLOR = {"baseline": "steelblue"}
_pal = ["indianred", "darkorange", "mediumpurple", "seagreen", "goldenrod"]
for i, a in enumerate([a for a in OUR_ARMS if a != "baseline"]):
    OUR_COLOR[a] = _pal[i % len(_pal)]

# Panel grid that stays readable however many games have runs (up to 27)
NCOLS = min(3, max(1, len(GAMES)))
NROWS = -(-len(GAMES) // NCOLS)

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(6 * NCOLS, 4.6 * NROWS),
                         squeeze=False)
for ax in axes.flat[len(GAMES):]:
    ax.set_visible(False)
for ax, g in zip(axes.flat, GAMES):
    k = GAME_KEYS[g]
    names, vals, cols = [], [], []
    for m in PUBLISHED:
        if m in BASELINES[k]:
            names.append(m); vals.append(hns(BASELINES[k][m], k)); cols.append("0.65")
    for a in OUR_ARMS:
        if a in RESULTS[g]:
            names.append(OUR_LABEL[a]); vals.append(hns(RESULTS[g][a]["mean"], k))
            cols.append(OUR_COLOR[a])
    x = np.arange(len(names))
    ax.bar(x, vals, color=cols)
    # seed spread on our arms
    for a in OUR_ARMS:
        if a in RESULTS[g]:
            i = names.index(OUR_LABEL[a])
            seed_h = [hns(v, k) for v in RESULTS[g][a]["means"]]
            ax.scatter([i] * len(seed_h), seed_h, color="k", s=12, zorder=3)
    ax.axhline(1.0, ls="--", c="gray", lw=1)
    ax.axhline(0.0, ls=":", c="gray", lw=1)
    top = max([1.06] + [v * 1.18 for v in vals])
    ax.text(len(names) - 0.5, 1.02, "human", ha="right", fontsize=9, color="gray")
    for xi, v in zip(x, vals):
        ax.text(xi, v + top * 0.012, f"{v:.3f}", ha="center", fontsize=8)
    ax.set_ylim(min(0, min(vals) * 1.2 if vals else 0) - 0.02, top)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=40, ha="right", fontsize=9)
    ax.set_title(k + ("" if BASELINES[k] else "  (not in the Atari-100k suite)"))
for row in axes:
    row[0].set_ylabel("human-normalised score (100k steps)")
fig.suptitle("Atari-100k: final-score HNS — published methods (grey) vs this repo's arms "
             "(dots = individual seeds)", y=1.0 + 0.04 / NROWS)
fig.tight_layout()
save_and_show(fig, "atari100k_hns_per_game")

## Aggregate over the config-selected suite games

Mean and median HNS over `AGG_GAMES` — the games whose config sets
`experiment.include_in_aggregate: true`, restricted to the 26-game suite and to games
with completed runs. Published methods are aggregated over the **same** games, so the
comparison is apples-to-apples on that subset; an arm missing runs on any selected game
is dropped from the chart (a mixed-subset aggregate would mislead). Anything short of
all 26 games is indicative — quote per-game numbers in preference to this chart.

In [ ]:
assert AGG_GAMES, ("no game with runs opts into the aggregate — set "
                   "experiment.include_in_aggregate: true in the game configs")
AGG_KEYS = [GAME_KEYS[g] for g in AGG_GAMES]

methods, cols, agg_mean, agg_med = [], [], [], []
for m in PUBLISHED:
    per = [hns(BASELINES[k][m], k) for k in AGG_KEYS]
    methods.append(m); cols.append("0.65")
    agg_mean.append(float(np.mean(per))); agg_med.append(float(np.median(per)))
for a in OUR_ARMS:
    missing = [g for g in AGG_GAMES if a not in RESULTS[g]]
    if missing:
        print(f"{OUR_LABEL[a]}: dropped from the aggregate — no runs on {missing}")
        continue
    per = [hns(RESULTS[g][a]["mean"], GAME_KEYS[g]) for g in AGG_GAMES]
    methods.append(OUR_LABEL[a]); cols.append(OUR_COLOR[a])
    agg_mean.append(float(np.mean(per))); agg_med.append(float(np.median(per)))

x = np.arange(len(methods))
fig, ax = plt.subplots(figsize=(max(10, 1.2 * len(methods)), 4.2))
b1 = ax.bar(x - 0.2, agg_mean, width=0.38, color=cols, label="mean HNS")
b2 = ax.bar(x + 0.2, agg_med, width=0.38, color=cols, alpha=0.55, label="median HNS")
for bs in (b1, b2):
    for b in bs:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.008,
                f"{b.get_height():.3f}", ha="center", fontsize=8)
ax.axhline(0, c="gray", lw=1)
ax.set_xticks(x); ax.set_xticklabels(methods, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("HNS")
if len(AGG_KEYS) == len(BENCHMARK_GAMES):
    scope = "the full 26-game Atari-100k suite"
elif len(AGG_KEYS) <= 4:
    scope = f"{' + '.join(AGG_KEYS)} ({len(AGG_KEYS)}-game subset — indicative only)"
else:
    scope = f"{len(AGG_KEYS)} config-selected suite games (subset — indicative only)"
ax.set_title(f"Aggregate HNS over {scope}")
ax.legend()
fig.tight_layout()
save_and_show(fig, "atari100k_hns_aggregate")

## Learning curves — raw training return vs environment steps

Training-episode raw scores (the sign-clipped reward is only what the agent optimises;
curves report `info["raw_reward"]` sums) against env steps, rolling mean per seed, then
seed-averaged per arm with a min–max band. The x-axis ends at the 100k budget.

In [ ]:
def rolling(x, w):
    x = np.asarray(x, float)
    return x if len(x) < w else np.convolve(x, np.ones(w) / w, mode="valid")

def curve_on_grid(runs, w=20, n=200):
    grid = np.linspace(0, ATARI100K_ENV_STEPS, n)
    stack = np.full((len(runs), n), np.nan)
    for i, r in enumerate(runs):
        steps = np.cumsum(r["steps"]) if r["steps"] is not None else None
        if steps is None or len(r["rewards"]) < w:
            continue
        y = rolling(r["rewards"], w)
        xs = steps[len(steps) - len(y):]
        inside = (grid >= xs[0]) & (grid <= xs[-1])
        stack[i, inside] = np.interp(grid[inside], xs, y)
    return grid, stack

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(6 * NCOLS, 4.2 * NROWS),
                         squeeze=False)
for ax in axes.flat[len(GAMES):]:
    ax.set_visible(False)
for ax, g in zip(axes.flat, GAMES):
    for arm in RESULTS[g]:
        grid, stack = curve_on_grid(RESULTS[g][arm]["runs"])
        m = np.nanmean(stack, axis=0)
        ax.plot(grid, m, color=OUR_COLOR[arm], label=OUR_LABEL[arm])
        ax.fill_between(grid, np.nanmin(stack, axis=0), np.nanmax(stack, axis=0),
                        color=OUR_COLOR[arm], alpha=0.15)
    ax.set_title(GAME_KEYS[g])
    ax.set_xlabel("environment steps (of 100k)")
for row in axes:
    row[0].set_ylabel("episode return (raw score, rolling-20)")
axes.flat[0].legend()
fig.suptitle("Training curves under the 100k budget (band = min–max across seeds)",
             y=1.0 + 0.03 / NROWS)
fig.tight_layout()
save_and_show(fig, "atari100k_learning_curves")

## Notes for the write-up

- Cite: benchmark protocol — Kaiser et al. 2020 (SimPLe); baseline numbers —
  Ye et al. 2021 Table 1 (EfficientZero); random/human references — Wang et al. 2016.
- The honest claim shape: *"under a DER-style double-DQN at replay ratio 1, adding the
  FHR penalty changes the 100k final score from X to Y on these games"* — the published
  methods contextualise the scale of the axis; they are not matched-recipe ablations.
- Every run's Hankel mechanism diagnostics (rank evolution, penalty traces, learned
  coefficients) are browsable per run in the result viewer (`python
  result_viewer_app/rank_viewer.py`), which groups these runs under the
  `fhrdqn100k_runs` manifest family with baseline vs exp<N> compare built in.